# Signature Verification — final demo

This is the finished model. Given two signatures it answers one question: **same hand, or a forgery?**

The short story of how we got here:
- **NB1** treated it as plain classification → looked perfect (0.999) but was cheating on a data leak.
- **NB2** switched to a Siamese network that actually *compares* two signatures → an honest 0.973.
- **NB3 / 3b / 3c** kept improving the tower. The winner is **3c**: a fine-tuned EfficientNet-B0
  trained with batch-hard mining (it focuses on the hardest skilled forgeries).

This notebook just **loads that trained model and shows it working** — no training here.

## Setup

In [ ]:
!git clone -q https://github.com/goyashek/Signature-forgery-verification.git

import os, json, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from keras.models import load_model

REPO = 'Signature-forgery-verification'
if not os.path.isdir(REPO):           # already inside the repo (running locally)
    REPO = '.'
DATA = os.path.join(REPO, 'sign_data_combined')

## Load the trained model

In [ ]:
# The tower ends in a Lambda layer that L2-normalizes the embedding. It was saved
# referencing a function named "l2", so we redefine it and hand it to load_model via
# custom_objects (safe_mode=False is also needed to allow Lambda deserialization).
import tensorflow as tf

def l2(t):
    return tf.math.l2_normalize(t, axis=1)

tower = load_model(os.path.join(REPO, 'models', 'siamese_bh_embedding.keras'),
                   custom_objects={'l2': l2}, safe_mode=False)
meta  = json.load(open(os.path.join(REPO, 'models', 'siamese_bh_meta.json')))

H, W = meta['img_h'], meta['img_w']
THRESHOLD = meta['global_threshold']      # distance below this = genuine (chosen at equal-error-rate)
print('model loaded:', meta['model'])
print(f'input {H}x{W}, decision threshold {THRESHOLD:.3f}')

## How it works

The model turns one signature into a 128-number "fingerprint" (an embedding). Two signatures
from the same hand land close together; a forgery lands farther away. We just measure the
distance between the two fingerprints and compare it to the threshold.

Preprocessing has to match training exactly: grayscale, resized, **inverted** (so the ink is
the signal and the white page is 0), then copied into 3 channels for EfficientNet.

In [ ]:
def load_sig(path):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    im = cv2.resize(im, (W, H))
    im = 255.0 - im.astype('float32')                 # invert: ink = signal, page = 0
    return np.repeat(im[..., None], 3, axis=2)        # 1 channel -> 3 for EfficientNet

def embed(path):
    return tower.predict(load_sig(path)[None], verbose=0)[0]

def verify(path1, path2):
    distance = np.linalg.norm(embed(path1) - embed(path2))
    verdict = 'GENUINE' if distance < THRESHOLD else 'FORGED'
    return verdict, distance

## Try it on real signatures

In [ ]:
import pandas as pd
m = pd.read_csv(os.path.join(DATA, 'manifest.csv'))

def files(writer, kind):
    sub = m[(m.writer == writer) & (m.kind == kind)]
    return [os.path.join(DATA, p) for p in sub.relpath]

def show(reference, query, title):
    verdict, d = verify(reference, query)
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].imshow(cv2.imread(reference, 0), cmap='gray'); ax[0].set_title('reference'); ax[0].axis('off')
    ax[1].imshow(cv2.imread(query, 0), cmap='gray');     ax[1].set_title('questioned'); ax[1].axis('off')
    fig.suptitle(f'{title}\nverdict: {verdict}   (distance {d:.2f}, threshold {THRESHOLD:.2f})')
    plt.tight_layout(); plt.show()

In [ ]:
# a writer the model never saw in training (test set: icdar >= 49)
w = 'icdar_049'
g = files(w, 'genuine'); f = files(w, 'forged')

show(g[0], g[1], 'genuine vs another genuine  (expect GENUINE)')
show(g[0], f[0], 'genuine vs forgery          (expect FORGED)')

In [ ]:
# and a Devanagari writer, to show it works across scripts
w = 'bhh_140'
g = files(w, 'genuine'); f = files(w, 'forged')

show(g[0], g[1], 'genuine vs another genuine  (expect GENUINE)')
show(g[0], f[0], 'genuine vs forgery          (expect FORGED)')

## Final results

All numbers are on **writer-independent test data** (people the model never saw in training)
with **leak-free pairs**, so they're trustworthy. The full progression:

| Notebook | Approach | Test AUC | Cross-dataset (NFI) AUC |
|---|---|---|---|
| NB1 | stacked-pair CNN | 0.999 *(data leak — not real)* | — |
| NB2 | Siamese CNN + contrastive | 0.973 | 0.791 |
| NB3 | SigNet tower + triplet | 0.915 | 0.852 |
| 3b | fine-tuned EfficientNet-B0 | 0.941 | 0.871 |
| **3c** | **+ batch-hard mining (this model)** | **0.986** | **0.877** |

**This model (3c), held-out test:**
- Overall: **AUC 0.986**, accuracy **93.85%**, FAR 6.52%, FRR 5.78%
- By script: Latin AUC 0.989, Devanagari AUC 0.984
- Best operating point (per-writer threshold): **AUC 0.989, FAR 3.48%, FRR 7.67%**

FAR = forgeries wrongly accepted, FRR = genuine wrongly rejected — the two errors that matter
for verification. 3c is the best model on every metric, in-domain and cross-dataset.

## The honest caveat

On a totally different dataset (NFI — different pens, scanners, forgery styles) the AUC drops
to 0.877 and, at the threshold tuned on our training data, it accepts too many forgeries. The
*ranking* is still good; it's the threshold that needs recalibrating per dataset/writer. This is
expected and well-documented in the signature literature — a single model score should never be
the only check in a real system.